# CEFR Hierarchical Bell-Curve Score

A **different** approach from the reshaping notebooks - here the bell curve comes from the
**models' own confidence**, with **no distribution reshaping** (no uniform/quantile transform).

## The idea

Two **hierarchical** binary models (a decision tree of classifiers):

```
                    all learners
                         |
             [Stage 1]  band 0  vs  {band 1, band 2}
                    /                     \
              band 0                  {band 1, band 2}
                                           |
                              [Stage 2]  band 1  vs  band 2
```

- **Stage 1** (`clf_low`): is the learner **band 0** or **upper** ({1,2})? -> `p_up = P(upper)`
- **Stage 2** (`clf_hi`): among the upper group, **band 1** or **band 2**? -> `p_two = P(band2 | upper)`
  (trained only on the band-1/band-2 subset)

**Why this gives a bell.** Band 1 vs band 2 is a *hard* distinction, so Stage 2's confidence is
often low (`p_two` near 0.5) - those learners land in the **middle**. We score using the
**log-odds (logit)** of the confidence, which is naturally **bell-distributed** (unlike the raw
probability, which piles at 0/1). Confident band-0 learners sit low, confident band-2 high, and
the uncertain middle fills in - a bell, straight out of the model, no reshaping.

**Score** = standardised confidence margin `logit(p_up) + p_up*logit(p_two)` placed on 0-100
(the **raw score**), then a **Beta(5,5) quantile reshaping** for a clean bell (the **beta
score**). We show raw vs Beta and the split points before/after.

## 0. Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             cohen_kappa_score, confusion_matrix, classification_report)
print("ready")

## 1. Load your data  <-- FILL THIS IN

In [ ]:
# TODO: assign your dataframe
df = None
# e.g. df = pd.read_csv("your_file.csv")

## 2. Columns  <-- FILL THIS IN

In [ ]:
FEATURE_COLS = []                 # one feature per group (8-11)

ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL = "ciid", "location", "split", "cefr"
TRAIN_VALUE, TEST_VALUE = "train", "test"
META_COLS = [ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL]

## 3. Configuration

In [ ]:
RANDOM_STATE = 42
BAND_MAP = {"A1": 0, "A2": 0, "B1": 1, "B2": 2, "C1": 2, "C2": 2}
BAND_NAMES = ["A1-A2", "B1", "B2-C1-C2"]
N_BANDS = 3

CALIBRATE = True      # calibrate the two classifiers -> smoother confidence -> cleaner bell
SPREAD = 16.0         # bell width: score = 50 +/- SPREAD * z(margin); ~+/-3 sd -> ~2..98
BASELINE_ACC, TARGET_ACC = 0.77, 0.82

## 4. Build train / test  (from the `split` column)

In [ ]:
assert df is not None and len(FEATURE_COLS) > 0, "Fill in df and FEATURE_COLS."
miss = [c for c in FEATURE_COLS + META_COLS if c not in df.columns]
assert not miss, f"missing columns: {miss}"

def to_band(s):
    s = pd.Series(s)
    if s.dtype.kind in "iuf" and set(pd.unique(s.dropna())) <= {0, 1, 2}:
        return s.astype(int).to_numpy()
    key = s.astype(str).str.strip().str.upper().str.replace(" ", "", regex=False)
    m = key.map(BAND_MAP); assert m.notna().all(), f"unmapped: {key[m.isna()].unique()}"
    return m.astype(int).to_numpy()

sp = df[SPLIT_COL].astype(str).str.strip().str.lower()
train_df, test_df = df.loc[sp == TRAIN_VALUE].copy(), df.loc[sp == TEST_VALUE].copy()
X_train, X_test = train_df[FEATURE_COLS].astype(float), test_df[FEATURE_COLS].astype(float)
y_train, y_test = to_band(train_df[LABEL_COL]), to_band(test_df[LABEL_COL])

print(f"train/test: {len(X_train)}/{len(X_test)} | dropped bad flags: {(~sp.isin([TRAIN_VALUE, TEST_VALUE])).sum()}")
print("train band counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("test  band counts:", dict(zip(*np.unique(y_test,  return_counts=True))))

## 5. The hierarchical model

Stage 1 (`clf_low`): band 0 vs upper, on all train rows.
Stage 2 (`clf_hi`): band 1 vs band 2, on the **upper subset** only.
Combine -> P(band0), P(band1), P(band2); predicted band = argmax.

In [ ]:
def make_clf():
    base = RandomForestClassifier(n_estimators=600, min_samples_leaf=3, max_features="sqrt",
                                  class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1)
    clf = CalibratedClassifierCV(base, cv=3, method="sigmoid") if CALIBRATE else base
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("clf", clf)])

# stage 1: band 0 vs upper
clf_low = make_clf().fit(X_train, (y_train >= 1).astype(int))
# stage 2: band 1 vs band 2 on the upper subset
sub = y_train >= 1
clf_hi = make_clf().fit(X_train[sub], (y_train[sub] == 2).astype(int))

def hier(X):
    p_up = clf_low.predict_proba(X)[:, 1]
    p_two = clf_hi.predict_proba(X)[:, 1]
    P = np.column_stack([1 - p_up, p_up * (1 - p_two), p_up * p_two])   # P0, P1, P2
    return p_up, p_two, P

p_up_tr, p_two_tr, P_tr = hier(X_train)
p_up_te, p_two_te, P_te = hier(X_test)
pred_tr, pred_te = P_tr.argmax(1), P_te.argmax(1)

y_full = np.concatenate([y_train, y_test]); pred_full = np.concatenate([pred_tr, pred_te])
print("=== hierarchical band accuracy ===")
print(f"  TRAIN {accuracy_score(y_train, pred_tr):.3f} | TEST {accuracy_score(y_test, pred_te):.3f} "
      f"| FULL {accuracy_score(y_full, pred_full):.3f}")
print(f"  test balanced {balanced_accuracy_score(y_test, pred_te):.3f} | "
      f"macroF1 {f1_score(y_test, pred_te, average='macro'):.3f} | "
      f"QWK {cohen_kappa_score(y_test, pred_te, weights='quadratic'):.3f}")
print(f"  (baseline {BASELINE_ACC:.0%}, target {TARGET_ACC:.0%})\n")
print("TEST confusion matrix:")
display(pd.DataFrame(confusion_matrix(y_test, pred_te, labels=[0, 1, 2]),
                     index=[f"true {b}" for b in BAND_NAMES], columns=[f"pred {b}" for b in BAND_NAMES]))

## 6. The score: raw confidence -> Beta reshaping

**Raw score** = standardised confidence margin
`margin = logit(p_up) + p_up * logit(p_two)`, placed on 0-100 (centred 50).

**Beta reshaping (quantile method):** map each score to its **percentile** (rank), then push
that through a symmetric **Beta(5,5)** - the same quantile-to-Beta method as the reshaping
notebook - for a clean, controlled bell. Two **split points** (cut-points) are tuned on the raw
score; because the Beta map is monotonic they move to new positions but keep the same bands. We
show **raw vs Beta** and **previous vs new split points**.

In [ ]:
from sklearn.preprocessing import QuantileTransformer
from scipy.stats import beta as _beta

def logit(p, eps=1e-3):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def margin(p_up, p_two):
    return logit(p_up) + p_up * logit(p_two)

# ---- RAW score: standardised confidence margin -> 0-100 ----
m_tr, m_te = margin(p_up_tr, p_two_tr), margin(p_up_te, p_two_te)
MU, SD = float(m_tr.mean()), float(m_tr.std() + 1e-9)
def to_score(m):
    return np.clip(50.0 + SPREAD * (np.asarray(m) - MU) / SD, 1.0, 99.0)
raw_tr, raw_te = to_score(m_tr), to_score(m_te)

# ---- split points on the RAW score (tuned on train vs the true bands) ----
def apply_cutpoints(s, t1, t2):
    s = np.asarray(s); return np.where(s <= t1, 0, np.where(s <= t2, 1, 2))
def fit_cutpoints(s, y, ngrid=120):
    s = np.asarray(s); cand = np.unique(np.percentile(s, np.linspace(0, 100, ngrid)))
    best_v, best = -1.0, (33.3, 66.7)
    for i in range(len(cand) - 1):
        for j in range(i + 1, len(cand)):
            v = accuracy_score(y, apply_cutpoints(s, cand[i], cand[j]))
            if v > best_v:
                best_v, best = v, (float(cand[i]), float(cand[j]))
    return best
prev_cuts = fit_cutpoints(raw_tr, y_train)

# ---- BETA reshaping (quantile method): rank -> Beta(5,5) ----
BETA_A = 5.0; _EPS = 1e-3
qt = QuantileTransformer(output_distribution="uniform", n_quantiles=min(len(raw_tr), 1000),
                         subsample=1000000000, random_state=RANDOM_STATE).fit(raw_tr.reshape(-1, 1))
def to_beta(s):
    p = np.clip(qt.transform(np.asarray(s, float).reshape(-1, 1)).ravel(), _EPS, 1 - _EPS)
    return 100.0 * _beta.ppf(p, BETA_A, BETA_A)
beta_tr, beta_te = to_beta(raw_tr), to_beta(raw_te)
new_cuts = tuple(float(x) for x in to_beta(np.array(prev_cuts)))   # monotonic -> same bands

raw_full  = np.concatenate([raw_tr, raw_te])
beta_full = np.concatenate([beta_tr, beta_te])
bnd = np.concatenate([pred_tr, pred_te])

print(f"split points  PREV (raw scale) : {prev_cuts[0]:.1f} / {prev_cuts[1]:.1f}")
print(f"split points  NEW  (Beta scale): {new_cuts[0]:.1f} / {new_cuts[1]:.1f}")
print("\nBeta score by band (full data):")
for b in range(N_BANDS):
    v = beta_full[bnd == b]
    if len(v):
        print(f"  {BAND_NAMES[b]:<9} {v.min():.0f}-{v.max():.0f}  median {np.median(v):.0f}  n={len(v)}")

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].hist(raw_full, bins=20, range=(0, 100), color="#c0553b")
    for c in prev_cuts: ax[0].axvline(c, color="k", ls="--")
    ax[0].set_title(f"raw score  (splits {prev_cuts[0]:.0f} / {prev_cuts[1]:.0f})")
    ax[1].hist(beta_full, bins=20, range=(0, 100), color="#3b6ea5")
    for c in new_cuts: ax[1].axvline(c, color="k", ls="--")
    ax[1].set_title(f"after Beta reshaping  (splits {new_cuts[0]:.0f} / {new_cuts[1]:.0f})")
    for a in ax:
        a.set_xlim(0, 100); a.set_xlabel("0-100 score"); a.set_ylabel("learners")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")

## 7. Final predictions - full dataset

Per learner: the two stage confidences, the three band probabilities, the **bell score**, and
the predicted band, with `ciid` / `split` / `region`.

In [ ]:
n_tr = len(X_train)
out = pd.DataFrame({
    ID_COL:   np.concatenate([train_df[ID_COL].values, test_df[ID_COL].values]),
    "region": np.concatenate([train_df[LOCATION_COL].values, test_df[LOCATION_COL].values]),
    "split":  ["train"] * n_tr + ["test"] * len(X_test),
    "true_band": [BAND_NAMES[i] for i in y_full],
    "p_up":  np.round(np.concatenate([p_up_tr, p_up_te]), 4),
    "p_two": np.round(np.concatenate([p_two_tr, p_two_te]), 4),
    "P0": np.round(np.concatenate([P_tr[:, 0], P_te[:, 0]]), 4),
    "P1": np.round(np.concatenate([P_tr[:, 1], P_te[:, 1]]), 4),
    "P2": np.round(np.concatenate([P_tr[:, 2], P_te[:, 2]]), 4),
    "raw_score":  np.round(raw_full, 2),
    "beta_score": np.round(beta_full, 2),
    "pred_band": [BAND_NAMES[i] for i in bnd],
})
with pd.option_context("display.max_rows", 400, "display.max_columns", 60):
    display(out)
# out.to_csv("hierarchical_predictions.csv", index=False)

## 8. Export: `hierarchical bell.csv`

Per learner: id, location, split, label, and the raw confidence 0-100 score, its Beta-reshaped
(bell) score, and the predicted band. Saved to `hierarchical bell.csv`.

In [ ]:
n_tr = len(X_train)
csv = pd.DataFrame({
    ID_COL:      np.concatenate([train_df[ID_COL].values, test_df[ID_COL].values]),
    "location":  np.concatenate([train_df[LOCATION_COL].values, test_df[LOCATION_COL].values]),
    "split":     ["train"] * n_tr + ["test"] * len(X_test),
    "label":     [BAND_NAMES[i] for i in y_full],
    "hier_raw":  np.round(raw_full, 2),
    "hier_bell": np.round(beta_full, 2),
    "hier_pred": [BAND_NAMES[i] for i in bnd],
})
csv.to_csv("hierarchical bell.csv", index=False)
print("saved 'hierarchical bell.csv' |", len(csv), "rows |", list(csv.columns))
with pd.option_context("display.max_rows", 400, "display.max_columns", 60):
    display(csv.head(10))

## Notes

- **Raw bell from confidence:** the raw score is already bell-ish because `logit(confidence)` is
  ~normal and Stage 2 (band1 vs band2) is genuinely uncertain. The **Beta reshaping** then makes
  it a clean, controlled bell (quantile -> Beta); the split points move but the bands are
  unchanged (monotonic).
- **Score is a continuous confidence measure**, so bands can overlap slightly on the score axis
  (honest uncertainty). The `pred_band` is the hierarchical argmax and is what to report as the
  class; `score` is the 0-100 presentation.
- **Knobs:** `SPREAD` (bell width), `CALIBRATE` (smoother confidence -> cleaner bell).
- Rigour upgrade: compute the train margins with out-of-fold predictions before standardising
  (here train stats are in-sample for simplicity; the test bell is unaffected by that choice).